# Otimização — capítulo 3: Otimização multidimensional sem restrições

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/ex03_2_3_nelder_mead.ipynb)

Corra a célula **Preparação** e depois o exemplo (no Colab: *Runtime → Run all*). No fim, o exemplo compara os resultados com os slides (*confere com os slides: sim*). Os slides usam vírgula decimal; aqui usa-se o ponto. Caderno gerado a partir de `ex03_2_3_nelder_mead.py` por `notebooks/make_notebooks.py`.

*Code comments and printed messages are in Portuguese.*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 3.2.3 Método do simplex de Nelder–Mead

Exemplo [`ex03_2_3_nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/ex03_2_3_nelder_mead.py) · funções da UC: [`nelder_mead.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_2_3_nelder_mead/nelder_mead.py)

```text
Reproduz os exemplos do deck 3.2.3 (Nelder–Mead).

1. Uma iteração à mão: f = x1^2 + x2^2, simplex (-1,0), (1,2), (-1,3).
2. Himmelblau a partir de (0,0), simplex (0,0), (1.2,0), (0,1.2),
   tolerâncias 1e-6: tabela k = 0..12; 54 iterações, n_f = 106, x* = (3,2).
   Com o simplex (0,0), (-1.2,0), (0,-1.2) chega-se a (-2.81; 3.13).
3. Rosenbrock a partir de (-1.5, 2): n_f até f < 1e-4 = 166 (simplex de
   arestas 0.5, o da comparação final em 3.3.3) e 139 (arestas de 5 %,
   o simplex inicial do deck).

O deck usa fminsearch (MATLAB) / scipy.optimize.minimize (Python); aqui
usa-se a implementação da UC, nelder_mead, com o pseudocódigo do deck.
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [2]:
import numpy as np

from nelder_mead import nelder_mead


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 3.2.3: método de Nelder–Mead")

sq = lambda x: x[0]**2 + x[1]**2
himmel = lambda x: (x[0]**2 + x[1] - 11)**2 + (x[0] + x[1]**2 - 7)**2
ros = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

# ------------------------------------------------------------ 1. à mão
print("\n1. Uma iteração à mão: f = x1^2 + x2^2, simplex (-1,0), (1,2), (-1,3)")
r = nelder_mead(sq, [[-1, 0], [1, 2], [-1, 3]], kmax=1, verbose=True)
S = r.simplex; Fs = [sq(x) for x in S]
print("novo simplex: (%g,%g), (%g,%g), (%g,%g) com f = %g, %g, %g;  avaliações nesta iteração: %d"
      % (*S[0], *S[1], *S[2], *Fs, r.nfev - 3))
c = [r.ops[1] == "reflexão", confere(S, [[-1, 0], [1, -1], [1, 2]], 4),
     confere(Fs, [1, 2, 5], 4), r.nfev - 3 == 1]
print("  reflexão aceite: %s | novo simplex: %s | f = 1, 2, 5: %s | 1 avaliação: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 2. Himmelblau
print("\n2. Himmelblau a partir de (0,0), simplex (0,0), (1.2,0), (0,1.2), tolx = tolf = 1e-6")
r = nelder_mead(himmel, [[0, 0], [1.2, 0], [0, 1.2]], tolx=1e-6, tolf=1e-6)
print("primeiras 13 linhas de info.history [k, n_f, f(x_b), f(x_w), x_b] e operação:")
print("%4s %5s %12s %12s %10s %10s  %s" % (*r.cols, "operação"))
for row, o in zip(r.history[:13], r.ops[:13]):
    print("%4d %5d %12.4f %12.4f %10.4f %10.4f  %s" % (*row, o))
print("... %d iterações, n_f = %d, x* = (%.6f, %.6f), f = %.1e  (%s)"
      % (r.nit, r.nfev, r.x[0], r.x[1], r.fx, r.message))
# tabela do slide (k = 0..12): operação, x_b, f(x_b), f(x_w)
Tops = ["inicial", "expansão", "reflexão", "reflexão", "contr. int.", "contr. int.", "contr. ext.",
        "contr. int.", "contr. int.", "contr. int.", "contr. int.", "contr. ext.", "contr. int."]
Txb = [[1.200, 0.000], [1.800, 1.800], [3.000, 0.600], [3.000, 0.600], [2.550, 1.650],
       [3.188, 1.762], [3.188, 1.762], [3.188, 1.762], [2.884, 1.921], [2.919, 2.050],
       [2.919, 2.050], [3.031, 1.983], [3.031, 1.983]]
Tfb = [125.0336, 39.3632, 15.2096, 15.2096, 11.0925, 1.3499, 1.3499, 1.3499, 0.7628,
       0.1972, 0.1972, 0.0303, 0.0303]
Tfw = [170.000, 126.954, 125.034, 39.363, 24.579, 15.210, 11.093, 2.965, 1.604, 1.350,
       0.763, 0.217, 0.197]
H = r.history[:13]
c = [r.ops[:13] == Tops, confere(H[:, 4:6], Txb, 3), confere(H[:, 2], Tfb, 4),
     confere(H[:, 3], Tfw, 3), r.nit == 54, r.nfev == 106, confere(r.x, [3, 2], 4), r.fx < 1e-10]
print("  tabela k = 0..12: operação: %s | x_b: %s | f(x_b): %s | f(x_w): %s\n"
      "  54 it.: %s | n_f = 106: %s | x* = (3,2): %s | f ~ 0: %s" % simnao(c))
ok = ok and all(c)

r2 = nelder_mead(himmel, [[0, 0], [-1.2, 0], [0, -1.2]])
print("simplex (0,0), (-1.2,0), (0,-1.2): x* = (%.2f; %.2f), f = %.1e" % (r2.x[0], r2.x[1], r2.fx))
c = [confere(r2.x, [-2.81, 3.13], 2)]
print("  chega a (-2.81; 3.13): %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------------------ 3. Rosenbrock
x0 = np.array([-1.5, 2.0])
print("\n3. Rosenbrock a partir de (-1.5, 2): n_f até f < 1e-4 (info.nhit)")
ra = nelder_mead(ros, [x0, x0 + [0.5, 0], x0 + [0, 0.5]], ftarget=1e-4)
rb = nelder_mead(ros, x0, ftarget=1e-4)          # simplex do deck: h_i = 0.05 max(1,|x0_i|)
print("simplex de arestas 0.5:     n_f até f < 1e-4 = %d  (final: %d it., n_f = %d, x = (%.4f, %.4f))"
      % (ra.nhit, ra.nit, ra.nfev, *ra.x))
print("simplex de arestas de 5 %%:  n_f até f < 1e-4 = %d  (final: %d it., n_f = %d, x = (%.4f, %.4f))"
      % (rb.nhit, rb.nit, rb.nfev, *rb.x))
c = [ra.nhit == 166, rb.nhit == 139]
print("  n_f = 166: %s | n_f = 139: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.2.3: método de Nelder–Mead

1. Uma iteração à mão: f = x1^2 + x2^2, simplex (-1,0), (1,2), (-1,3)
   k   n_f       f(x_b)       f(x_w)       x_b1       x_b2  operação
   0     3       1.0000      10.0000    -1.0000     0.0000  inicial
   1     4       1.0000       5.0000    -1.0000     0.0000  reflexão
novo simplex: (-1,0), (1,-1), (1,2) com f = 1, 2, 5;  avaliações nesta iteração: 1
  reflexão aceite: sim | novo simplex: sim | f = 1, 2, 5: sim | 1 avaliação: sim

2. Himmelblau a partir de (0,0), simplex (0,0), (1.2,0), (0,1.2), tolx = tolf = 1e-6
primeiras 13 linhas de info.history [k, n_f, f(x_b), f(x_w), x_b] e operação:
   k   n_f       f(x_b)       f(x_w)       x_b1       x_b2  operação
   0     3     125.0336     170.0000     1.2000     0.0000  inicial
   1     5      39.3632     126.9536     1.8000     1.8000  expansão
   2     7      15.2096     125.0336     3.0000     0.6000  reflexão
   3     8      15.2096      39.3632     3.0000     0.6000  reflexão
   4